# 04 — Model Training & Evaluation
Train LightGBM on the feature set, evaluate performance, analyse feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path

DATA   = Path('../data')
MODELS = Path('../models')
VIZ    = Path('../visualizations/model')
VIZ.mkdir(parents=True, exist_ok=True)

In [ ]:
features = pd.read_csv(DATA / 'features.csv', parse_dates=['hour'])
print(f'Loaded: {features.shape}')
features.head()

## Prepare Features & Target

In [ ]:
TARGET = 'avg_occupancy_rate'

DROP = ['hour', 'region', 'date', 'holiday_name', 'holiday_type',
        'event_genres', 'event_count']

FEATURE_COLS = [
    c for c in features.columns
    if c not in DROP + [TARGET]
    and features[c].dtype in ['float64', 'int64', 'bool']
]

print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
df = features.dropna(subset=[TARGET] + FEATURE_COLS).copy()
for col in FEATURE_COLS:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## Train LightGBM

In [ ]:
params = {
    'objective':        'regression',
    'metric':           'rmse',
    'num_leaves':       63,
    'learning_rate':    0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'min_child_samples': 20,
    'verbose':          -1,
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_test,  label=y_test, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=50)
    ]
)
print(f'Best iteration: {model.best_iteration}')

## Evaluate

In [ ]:
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

rmse = mean_squared_error(y_test, y_pred, squared=False)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual Occupancy Rate')
axes[0].set_ylabel('Predicted Occupancy Rate')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})')

residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.savefig(VIZ / 'actual_vs_predicted.png', dpi=150)
plt.show()

## Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature':    model.feature_name(),
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=True).tail(20)

plt.figure(figsize=(10, 7))
plt.barh(importance['feature'], importance['importance'], color='steelblue', alpha=0.85)
plt.title('Top 20 Feature Importances (Gain)')
plt.xlabel('Importance (Gain)')
plt.tight_layout()
plt.savefig(VIZ / 'feature_importance.png', dpi=150)
plt.show()

## Save Model

In [ ]:
import pickle

MODELS.mkdir(exist_ok=True)
model_path = MODELS / 'parking_demand_lgbm.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

perf = pd.DataFrame([{'rmse': rmse, 'mae': mae, 'r2': r2,
                       'n_train': len(X_train), 'n_test': len(X_test),
                       'best_iteration': model.best_iteration}])
perf.to_csv(MODELS / 'performance/final_model_performance.csv', index=False)
print(f'Model saved to {model_path}')
print(perf)